BASIC SET UP

In [ ]:
!nvidia-smi

Tue Oct 13 09:03:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/story_narrator'
for sub in ['data', 'models', 'audio', 'outputs']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print('Project folder ready:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder ready: /content/drive/MyDrive/story_narrator


In [ ]:
!pip install -q coqui-tts "transformers<5" noisereduce librosa soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the p

In [ ]:
import os, glob, transformers
os.environ['COQUI_TOS_AGREED'] = '1'
REF_RAW = glob.glob(f'{BASE}/audio/*')[0]
print(REF_RAW)
print(transformers.__version__)
from transformers.pytorch_utils import isin_mps_friendly
print('import ok')

/content/drive/MyDrive/story_narrator/audio/Recording.m4a
4.57.6
import ok


TEST VOICE CLONING

In [ ]:
import os, shutil
os.environ['COQUI_TOS_AGREED'] = '1'

from google.colab import files
uploaded = files.upload()
clip_name = list(uploaded.keys())[0]
REF_RAW = f'{BASE}/audio/{clip_name}'
shutil.copy(clip_name, REF_RAW)
print('Saved to', REF_RAW)

Saving Recording.m4a to Recording (1).m4a
Saved to /content/drive/MyDrive/story_narrator/audio/Recording (1).m4a


In [ ]:


import os, glob
os.environ['COQUI_TOS_AGREED'] = '1'
BASE = '/content/drive/MyDrive/story_narrator'
REF_RAW = glob.glob(f'{BASE}/audio/*')[0]
print(REF_RAW)

/content/drive/MyDrive/story_narrator/audio/Recording.m4a


In [ ]:
import torch
from TTS.api import TTS
from IPython.display import Audio

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

test_out = f'{BASE}/outputs/test_clone.wav'
tts.tts_to_file(
    text='Once upon a time, in a little village by the sea, there lived a curious girl named Mira.',
    speaker_wav=REF_RAW,
    language='en',
    file_path=test_out,
)
Audio(test_out)

AUDIO PREPROCESSING

In [ ]:
import numpy as np, librosa, soundfile as sf, noisereduce as nr

TARGET_SR = 22050
MIN_SEC = 6      # reject clips shorter than this (after trimming silence)
MAX_SEC = 30     # use at most this much audio

def preprocess_voice(in_path, out_path, target_sr=TARGET_SR):
    """Returns a list of warnings. Raises ValueError if the clip is unusable."""
    warnings = []
    y, _ = librosa.load(in_path, sr=target_sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)

    duration = len(y) / target_sr
    if duration < MIN_SEC:
        raise ValueError(f'Clip is too short ({duration:.1f}s of speech). Please upload at least {MIN_SEC} seconds.')
    if duration > MAX_SEC:
        y = y[: MAX_SEC * target_sr]
        warnings.append(f'Clip was longer than {MAX_SEC}s, so only the first {MAX_SEC}s were used.')

    # rough noise check: compare loud frames to quiet frames
    rms = librosa.feature.rms(y=y)[0]
    ratio_db = 20 * np.log10((np.percentile(rms, 90) + 1e-8) / (np.percentile(rms, 10) + 1e-8))
    if ratio_db < 10:
        warnings.append('The clip seems noisy or has constant background sound. Results may be worse.')

    y = nr.reduce_noise(y=y, sr=target_sr, stationary=True, prop_decrease=0.8)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak * 0.95
    sf.write(out_path, y, target_sr)
    return warnings

REF_CLEAN = f'{BASE}/audio/reference_clean.wav'
print(preprocess_voice(REF_RAW, REF_CLEAN))
Audio(REF_CLEAN)

/tmp/ipykernel_5278/3060709520.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(in_path, sr=target_sr, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[]


In [ ]:
# Clone again using the cleaned clip and compare it with the first test
tts.tts_to_file(
    text='Once upon a time, in a little village by the sea, there lived a curious girl named Mira.',
    speaker_wav=REF_CLEAN, language='en',
    file_path=f'{BASE}/outputs/test_clone_clean.wav',
)
Audio(f'{BASE}/outputs/test_clone_clean.wav')

FINE TUNE LLM AND PREPARE STORY DATA

In [ ]:
!pip install -q "transformers<5" peft datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00


In [ ]:
import re, random, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
random.seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

GENRES = ['bedtime story', 'animal tale', 'adventure', 'magical story']

def make_prompt(genre, theme, characters, length):
    return (f'### Instruction:\nWrite a {length} {genre} for children.\n'
            f'Characters: {characters}\nTheme: {theme}\n\n### Story:\n')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
THEME_WORDS = {
    'friendship': ['friend', 'together', 'play'],
    'kindness': ['kind', 'help', 'share'],
    'bravery': ['brave', 'afraid', 'scared', 'courage'],
    'curiosity': ['curious', 'explore', 'wonder'],
    'honesty': ['truth', 'lie', 'honest', 'sorry'],
}
GENRE_WORDS = {
    'animal tale': ['dog', 'cat', 'bird', 'rabbit', 'bear', 'fox', 'duck'],
    'adventure': ['adventure', 'journey', 'forest', 'cave', 'map'],
    'magical story': ['magic', 'fairy', 'wizard', 'dragon', 'princess'],
}
NOT_NAMES = {'The', 'She', 'He', 'They', 'One', 'Once', 'But', 'It', 'Her', 'His', 'Then',
             'When', 'After', 'Every', 'There', 'So', 'And', 'As', 'Now', 'We', 'I', 'You',
             'That', 'This', 'What', 'Why', 'How', 'Yes', 'No', 'Soon', 'Later', 'Suddenly'}

def label_story(text):
    lower = text.lower()
    theme = next((t for t, ws in THEME_WORDS.items() if any(w in lower for w in ws)), 'kindness')
    genre = next((g for g, ws in GENRE_WORDS.items() if any(w in lower for w in ws)), 'bedtime story')
    caps = [w for w in re.findall(r'\b[A-Z][a-z]{2,}\b', text) if w not in NOT_NAMES]
    counts = {}
    for w in caps:
        counts[w] = counts.get(w, 0) + 1
    names = [w for w, c in sorted(counts.items(), key=lambda x: -x[1]) if c >= 2][:2]
    characters = ', '.join(names) if names else 'a child'
    length = 'short' if len(text.split()) < 150 else 'medium'
    return genre, theme, characters, length

raw = load_dataset('roneneldan/TinyStories', split='train[:30000]')

def build(example):
    story = example['text'].strip()
    genre, theme, characters, length = label_story(story)
    return {'text': make_prompt(genre, theme, characters, length) + story + tokenizer.eos_token}

data = raw.map(build, remove_columns=raw.column_names)
print(data[0]['text'][:600])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

### Instruction:
Write a short bedtime story for children.
Characters: Lily
Theme: friendship

### Story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for 


In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=384)

tokenized = data.map(tokenize, batched=True, remove_columns=['text'])
split = tokenized.train_test_split(test_size=500, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(len(train_ds), len(eval_ds))

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

29500 500


In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={'': 0})
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f'{BASE}/models/checkpoints',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500,
    warmup_steps=30,
    lr_scheduler_type='cosine',
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    report_to='none',
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
# If the session disconnected earlier, re-run the cells above and use: trainer.train(resume_from_checkpoint=True)
trainer.train()

Step,Training Loss
25,1.623600
50,1.318600
75,1.271600
100,1.257900
125,1.238900
150,1.224800
175,1.218300
200,1.229100
225,1.186800
250,1.197500


TrainOutput(global_step=500, training_loss=1.226704433441162, metrics={'train_runtime': 2670.1423, 'train_samples_per_second': 2.996, 'train_steps_per_second': 0.187, 'total_flos': 1.911846905393971e+16, 'train_loss': 1.226704433441162, 'epoch': 0.2711864406779661})

In [ ]:
import math
metrics = trainer.evaluate()
print('Eval loss:', metrics['eval_loss'], '| Perplexity:', math.exp(metrics['eval_loss']))

ADAPTER_DIR = f'{BASE}/models/story_lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to', ADAPTER_DIR)

Eval loss: 1.1666314601898193 | Perplexity: 3.211157487621688
Saved adapter to /content/drive/MyDrive/story_narrator/models/story_lora


BUILDING STORY GENERATOR

In [ ]:
import gc
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={'': 0})
model = PeftModel.from_pretrained(base, f'{BASE}/models/story_lora')
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
import textwrap

def generate_story(genre, theme, characters, length='short', temperature=0.8, max_new_tokens=350):
    prompt = make_prompt(genre, theme, characters, length)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=0.9, repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    # drop an unfinished last sentence if generation was cut off
    if text and text[-1] not in '.!?"':
        cut = max(text.rfind('.'), text.rfind('!'), text.rfind('?'))
        if cut > 0:
            text = text[:cut + 1]
    return text

def split_segments(text, max_chars=220):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    segments, current = [], ''
    for s in sentences:
        pieces = textwrap.wrap(s, max_chars) if len(s) > max_chars else [s]
        for p in pieces:
            if len(current) + len(p) + 1 <= max_chars:
                current = (current + ' ' + p).strip()
            else:
                if current:
                    segments.append(current)
                current = p
    if current:
        segments.append(current)
    return segments

story = generate_story('animal tale', 'friendship', 'a fox named Rufus, a rabbit named Pip')
print(story)
print()
print(split_segments(story))

One day, a fox named Rufus wanted to get food. He saw a small bunny named Pip eating grass. Rufus was hungry and wanted to eat the same thing as Pip.

Rufus said to Pip, "Hi! I want some of your hay." But Pip didn't like Rufus. He thought Rufus might be too greedy. So Pip told Rufus, "No, thank you. My hay is enough for me."

But Rufus didn't give up. He asked again and again until finally Pip gave in. Rufus took a big bite of the hay, but it made him sick. He had trouble moving his legs and soon he lay down on the ground.

The moral of this story is that it's important to listen when people say no, even if they don't mean it.

['One day, a fox named Rufus wanted to get food. He saw a small bunny named Pip eating grass. Rufus was hungry and wanted to eat the same thing as Pip. Rufus said to Pip, "Hi!', 'I want some of your hay." But Pip didn\'t like Rufus. He thought Rufus might be too greedy. So Pip told Rufus, "No, thank you. My hay is enough for me."\n\nBut Rufus didn\'t give up.', 

FULL PIPELINE CONNECT

In [ ]:
def narrate(story_text, voice_path, out_path, pause_sec=0.4):
    segments = split_segments(story_text)
    sr = tts.synthesizer.output_sample_rate
    pause = np.zeros(int(sr * pause_sec), dtype=np.float32)
    chunks = []
    for seg in segments:
        wav = tts.tts(text=seg, speaker_wav=voice_path, language='en')
        chunks.append(np.array(wav, dtype=np.float32))
        chunks.append(pause)
    sf.write(out_path, np.concatenate(chunks), sr)
    return out_path

def story_to_audio(genre, theme, characters, length, voice_path, out_path):
    story = generate_story(genre, theme, characters, length)
    narrate(story, voice_path, out_path)
    return story, out_path

story, wav_path = story_to_audio('bedtime story', 'kindness', 'a little girl named Mira, a sleepy owl',
                                 'short', REF_CLEAN, f'{BASE}/outputs/story_1.wav')
print(story)
Audio(wav_path)

Once upon a time, there was a little girl named Mira. She had a very sleepy owl who lived in her closet. One day, the owl woke up and asked Mira if she could take a nap. Mira said yes and the owl went to sleep.

But then, something bad happened. A big storm came and blew out all of Mira's light bulbs! Suddenly, the owl woke up and saw that it couldn't see anymore. It felt sad because it didn't have any light to help him rest.

Mira noticed that the owl wasn't feeling good and gave him some warm blankets to keep him cozy. The owl soon fell asleep again and slept through the storm. From that day on, the owl always kept an eye on Mira's closet so nothing like that would happen again.


SYSTEM EVALUATION

In [ ]:
!pip install -q jiwer speechbrain "transformers<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 33.1 MB/s eta 0:00:00


In [ ]:
import jiwer
from transformers import pipeline

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
spk_model = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir=f'{BASE}/models/spkrec', run_opts={'device': dev})
asr = pipeline('automatic-speech-recognition', model='openai/whisper-small', device=0 if dev == 'cuda' else -1)

def embed(path):
    y, _ = librosa.load(path, sr=16000, mono=True)
    t = torch.tensor(y).unsqueeze(0).to(dev)
    return spk_model.encode_batch(t).squeeze()

def speaker_similarity(ref_path, gen_path):
    return torch.nn.functional.cosine_similarity(embed(ref_path), embed(gen_path), dim=0).item()

def word_error_rate(audio_path, reference_text):
    hyp = asr(audio_path, chunk_length_s=30)['text']
    norm = lambda s: re.sub(r'[^a-z0-9\s]', '', s.lower()).strip()
    return jiwer.wer(norm(reference_text), norm(hyp))

print('Speaker similarity:', round(speaker_similarity(REF_CLEAN, wav_path), 3))
print('WER:', round(word_error_rate(wav_path, story), 3))

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Speaker similarity: 0.627


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


WER: 0.029


GRADIO INTERFACE

In [ ]:

import os, tempfile, inspect
import numpy as np, soundfile as sf
import gradio as gr

try:
    demo.close()
except Exception:
    pass

THEMES = ['kindness', 'friendship', 'bravery', 'curiosity', 'honesty']
RAG_READY = 'generate_story_v2' in globals()


def write_story(genre, theme, characters, length, creativity, use_rag, progress=gr.Progress()):
    if not characters or not characters.strip():
        raise gr.Error('Please describe at least one character.')
    progress(0.2, desc='Writing your story...')
    theme = theme or 'kindness'
    if RAG_READY and use_rag:
        story, used = generate_story_v2(genre, theme, characters.strip(), length,
                                        use_rag=True, temperature=creativity)
    else:
        story = generate_story(genre, theme, characters.strip(), length, temperature=creativity)
        used = 'Story library was not used.'
    if not story:
        raise gr.Error('The story came out empty. Please try again.')
    return story, (used or 'Story library was not used.')


def narrate_story(story_text, voice1, voice2, consent, progress=gr.Progress()):
    if not consent:
        raise gr.Error('Please confirm that you have permission from this person to use their voice.')
    if not story_text or len(story_text.strip()) < 20:
        raise gr.Error('Please write or paste a story first.')
    voices = [v for v in (voice1, voice2) if v]
    if not voices:
        raise gr.Error('Please upload or record a voice clip.')

    tmp_dir = tempfile.mkdtemp()
    clean, notes = [], []
    try:
        for i, v in enumerate(voices):
            p = os.path.join(tmp_dir, f'voice_{i}.wav')
            try:
                notes += preprocess_voice(v, p)
            except ValueError as e:
                raise gr.Error(str(e))
            clean.append(p)
        progress(0.1, desc='Voice prepared')

        segments = split_segments(story_text)
        sr = tts.synthesizer.output_sample_rate
        pause = np.zeros(int(sr * 0.4), dtype=np.float32)
        parts = []
        for i, seg in enumerate(segments):
            progress(0.1 + 0.85 * i / len(segments), desc=f'Narrating part {i + 1} of {len(segments)}')
            wav = tts.tts(text=seg, speaker_wav=clean if len(clean) > 1 else clean[0], language='en')
            parts.append(np.array(wav, dtype=np.float32))
            parts.append(pause)
        out_path = os.path.join(tmp_dir, 'story.wav')
        sf.write(out_path, np.concatenate(parts), sr)
    finally:
        for p in clean:
            if os.path.exists(p):
                os.remove(p)

    status = '✅ Your narrated story is ready. Press play, or use the download button on the player.'
    if notes:
        status += '\n\n⚠️ ' + ' '.join(sorted(set(notes)))
    return out_path, status


CSS = """
.hero {text-align:center; padding:20px 12px; border-radius:16px;
       background:linear-gradient(135deg,#6d5dfc,#c86dd7); color:white;}
.hero h1 {margin:0; font-size:2rem; color:white;}
.hero p {margin:8px 0 0 0; opacity:0.95; color:white;}
.footer-note {text-align:center; opacity:0.75; font-size:0.9rem;}
"""
THEME = gr.themes.Soft(primary_hue='violet', secondary_hue='pink')

# Works with both older and newer Gradio versions
blocks_kw = {}
if 'theme' in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kw = dict(theme=THEME, css=CSS)

with gr.Blocks(title='AI Story Narrator', **blocks_kw) as demo:
    gr.HTML('<div class="hero"><h1>📖 AI Story Narrator</h1>'
            '<p>Create a story, then hear it narrated in the voice of someone you love.</p></div>')

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown('### 1. Choose your story')
            genre = gr.Dropdown(GENRES, value='bedtime story', label='Genre')
            theme = gr.Dropdown(THEMES, value='kindness', label='Theme (*Customisable)',
                                allow_custom_value=True)
            characters = gr.Textbox(value='a little girl named Mira, a sleepy owl', label='Characters',
                                    placeholder='e.g. a brave boy named Arjun, a talking parrot')
            length = gr.Radio(['short', 'medium'], value='short', label='Length')
            creativity = gr.Slider(0.5, 1.1, value=0.8, step=0.05, label='Creativity',
                                   info='Lower is safer and simpler, higher is more imaginative.')
            use_rag = gr.Checkbox(value=False, visible=RAG_READY,
                                  label='Use the fairy tale library for inspiration (RAG)')
            btn_story = gr.Button('✍️ Write my story', variant='primary')

        with gr.Column(scale=6):
            gr.Markdown('### 2. Read and edit your story')
            story_box = gr.Textbox(lines=14, interactive=True,
                                   label='Story (*Customisable before narration)')
            with gr.Accordion('Library passage used for inspiration', open=False, visible=RAG_READY):
                inspiration_box = gr.Textbox(lines=4, interactive=False, label='')

    gr.Markdown('### 3. Choose the narrator voice')
    with gr.Row():
        voice1 = gr.Audio(sources=['upload', 'microphone'], type='filepath',
                          label='Voice clip (10 to 30 seconds)')
        voice2 = gr.Audio(sources=['upload', 'microphone'], type='filepath',
                          label='Optional second clip (helps keep the accent)')
    consent = gr.Checkbox(label='I have permission from this person to use their voice')
    btn_narrate = gr.Button('🎙️ Narrate my story', variant='primary')
    status = gr.Markdown()
    audio_out = gr.Audio(label='Narrated story (AI-generated voice)', type='filepath')

    gr.Markdown('The narration is AI-generated. Uploaded voice clips are deleted right after use '
                'and are not stored.', elem_classes=['footer-note'])

    btn_story.click(write_story, [genre, theme, characters, length, creativity, use_rag],
                    [story_box, inspiration_box])
    btn_narrate.click(narrate_story, [story_box, voice1, voice2, consent], [audio_out, status])

launch_kw = {}
if 'theme' in inspect.signature(demo.launch).parameters and not blocks_kw:
    launch_kw = dict(theme=THEME, css=CSS)

demo.queue()
demo.launch(share=True, **launch_kw)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://<temporary-gradio-link>

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
